# Day 2 (teammate B) - ISL words (needs the Day-1 output dataset)
Inputs: `slr-code` and the Day-1 output dataset (`include50/` inside). Internet ON. Set the accelerator to **GPU T4** before the training cell.

In [ ]:
# Finds the uploaded "slr-code" dataset wherever Kaggle mounts it (the path differs between Kaggle versions).
import glob, os, shutil, subprocess, sys, zipfile
ON_KAGGLE = os.path.exists("/kaggle/input")
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
CODE = f"{WORK}/code"
hits = glob.glob("/kaggle/input/**/slr/__init__.py", recursive=True) + glob.glob("/content/slr_code/**/slr/__init__.py", recursive=True)
if hits:
    shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE, dirs_exist_ok=True)
else:
    zips = glob.glob("/kaggle/input/**/slr_code.zip", recursive=True) + glob.glob("/content/**/slr_code.zip", recursive=True)
    if zips:
        zipfile.ZipFile(zips[0]).extractall(CODE)
    else:
        print("slr-code NOT FOUND. Add it: right sidebar -> + Add Input -> Your Datasets -> slr-code. Currently attached:")
        for p in sorted(glob.glob("/kaggle/input/*") + glob.glob("/kaggle/input/*/*")):
            print("  ", p)
        raise SystemExit("Attach the slr-code dataset, then run this cell again.")
os.chdir(CODE)
print("code ready in", CODE, os.listdir(CODE))
# mediapipe 0.10.14 still has the `mp.solutions` API we use. If pip complains about numpy, run
# `!pip install "numpy<2"` and restart the session once.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mediapipe==0.10.14", "remotezip", "opencv-python-headless"])

## 1. Point at the Day-1 keypoints
Edit `INCLUDE_KP`: the sidebar path of the Day-1 dataset + `/include50`. It should contain `train`, `val`, `test` folders.

In [ ]:
INCLUDE_KP = "/kaggle/input/REPLACE-WITH-DAY1-OUTPUT-DATASET/include50"
!ls $INCLUDE_KP && ls $INCLUDE_KP/train | wc -l

## 2. Train (GPU T4, about 5-15 minutes)

In [ ]:
!python scripts/train_words.py --data $INCLUDE_KP --out $WORK/models/isl_words.pt --epochs 150 --seeds 3

## 3. Package for download (a few MB)

In [ ]:
!cd $WORK && zip -r isl_models.zip models && ls -lh isl_models.zip

Then **Save Version -> Save & Run All**, open the Output tab and download `isl_models.zip`.